This is a test of the new data / background preparation process.

Let's start by downloading some test data

In [1]:
from gdt.missions.fermi.time import Time
from gdt.missions.fermi.gbm.finders import ContinuousFinder

t0 = 524666469.44569993
finder = ContinuousFinder(Time(t0, format='fermi'))
finder.get_tte(".")

Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

[PosixPath('glg_tte_b0_170817_12z_v00.fit.gz'),
 PosixPath('glg_tte_b1_170817_12z_v00.fit.gz'),
 PosixPath('glg_tte_n0_170817_12z_v00.fit.gz'),
 PosixPath('glg_tte_n1_170817_12z_v00.fit.gz'),
 PosixPath('glg_tte_n2_170817_12z_v00.fit.gz'),
 PosixPath('glg_tte_n3_170817_12z_v00.fit.gz'),
 PosixPath('glg_tte_n4_170817_12z_v00.fit.gz'),
 PosixPath('glg_tte_n5_170817_12z_v00.fit.gz'),
 PosixPath('glg_tte_n6_170817_12z_v00.fit.gz'),
 PosixPath('glg_tte_n7_170817_12z_v00.fit.gz'),
 PosixPath('glg_tte_n8_170817_12z_v00.fit.gz'),
 PosixPath('glg_tte_n9_170817_12z_v00.fit.gz'),
 PosixPath('glg_tte_na_170817_12z_v00.fit.gz'),
 PosixPath('glg_tte_nb_170817_12z_v00.fit.gz')]

Let's also define some search settings. These are things that could go into a settings class.

In [2]:
import numpy as np
from gdt.missions.fermi.gbm.detectors import GbmDetectors

nai_edges = np.array([0, 8, 20, 33, 51, 85, 106, 127, 128])
bgo_edges = np.array([0, 8, 21, 40, 65, 90, 112, 124, 128])

settings = {
    'win_width': 60,
    'min_loglr': 5,
    'min_dur': 0.064, 'max_dur': 8.192,
    'min_step': 0.064,'num_steps': 8,
    'detectors':
        {det.name: {'channel_edges': nai_edges, 'search_channels': [1, 2, 3, 4, 5, 6]} for det in GbmDetectors.nai()} |
        {det.name: {'channel_edges': bgo_edges, 'search_channels': [0, 1, 2, 3, 4, 5, 6, 7]} for det in GbmDetectors.bgo()},
}

# these items could be properties / mapping methods for a settings class
detectors = list(settings['detectors'].keys())
channel_edges = {det: settings['detectors'][det]['channel_edges'] for det in detectors}
search_channels = {det: settings['detectors'][det]['search_channels'] for det in detectors}
time_range = np.array([-1, 1]) * max([0.5 * settings['win_width'] + settings['max_dur'] + 1.024, 30])
bkgd_range = [-40, 40]
phaii_resolution = settings['min_dur']
channel_mask = np.ravel(
    [[channel in search_channels[det] for channel in range(len(channel_edges[det]) - 1)] for det in detectors])

Now let's open the TTE data and update its trigtime to the time we want to search. This way all future times can be computed relative to this time of interest. We'll probably want a better way to handle this in the future using astropy.time objects.

During this step we'll also rebin the energy of the TTE files so that all future data / background formats that derive from it can inherit the same energy binning.

Store TTE data in a DataCollection for easier manipulation during future steps.

In [3]:
from rich.progress import track
from data import update_tte_trigtime
from gdt.core.collection import DataCollection
from gdt.core.binning.binned import rebin_by_edge_index
from gdt.missions.fermi.gbm.tte import GbmTte

tte_data = []
for det in track(detectors, description="Opening TTE files"):
    path = f"glg_tte_{det}_170817_12z_v00.fit.gz"
    tte = update_tte_trigtime(GbmTte.open(path), t0)
    tte = tte.rebin_energy(rebin_by_edge_index, channel_edges[det])
    tte_data.append(tte)

ttes = DataCollection.from_list(tte_data, names=detectors)

Output()

Now let's pre-bin the TTE data in time using the minimum step resolution from the search settings. This will allow us to quickly sum time bins when performing the search itself.

In [4]:
from gdt.core.binning.unbinned import bin_by_time

phaiis = DataCollection.from_list(
    ttes.to_phaii(bin_by_time, phaii_resolution, time_ref=0, time_range=time_range),
    names=detectors)

Next we'll pass the Phaii data to a **CountMatrix** class. The goal of this class is to act as an interface between the underlying data format (phaii) and the format needed by the likelihood method of the search (1D matrix of counts for all detectors and energy bins)

Notes:
1. CountMatrix accepts collections of either Phaii or TTE. TTE data can be useful in cases where we can't pre-bin the data in time, such as the multi-mission search.
2. this could probably be consolidated with the previous step.

In [5]:
from data import CountMatrix

data = CountMatrix(phaiis)
counts, exposure = data.counts(1.728, 2.240)

print("\nData:")
print("  counts", counts.reshape((len(detectors), 8)))
print("  exposure", exposure)


Data:
  counts [[ 38. 179. 119.  91.  96.  20.  14.  54.]
 [ 30. 201. 135. 111. 106.  17.  19.  42.]
 [ 35. 189. 125. 113. 108.  25.  37.  24.]
 [ 43. 207. 112.  93.  80.  23.  15.  30.]
 [ 31. 195. 131. 111.  85.  14.  40.  19.]
 [ 49. 242. 145. 126. 113.  19.  30.  14.]
 [ 34. 149.  90.  77.  71.  18.  30.  14.]
 [ 47. 194. 121.  91.  76.  14.  13.  42.]
 [ 41. 172. 109.  75.  73.  14.  32.  19.]
 [ 29. 121. 114.  81.  84.  23.  55.   6.]
 [ 22.  61.  73. 106.  60.  18.  37.  14.]
 [ 25.  79.  66. 111.  83.  28.  18.  40.]
 [257. 131. 207.  82.  15.   6.  10.  57.]
 [240. 137. 171.  66.  23.  17.  10.  55.]]
  exposure [0.5100118 0.5099706 0.5101168 0.5102102 0.5102318 0.5099776 0.5106406
 0.5101344 0.5104684 0.5106218 0.5108798 0.510534  0.5095892 0.5097236]


Next we'll fit the background using a polynomial fit applied to phaii data. This can easily be swapped for a NaisePoisson sliding window fit to TTE data.

In [6]:
from gdt.core.background.fitter import BackgroundFitter
from gdt.core.background.binned import Polynomial

backfitters = DataCollection.from_list(
    [BackgroundFitter.from_phaii(phaii, Polynomial, time_ranges=[bkgd_range]) for phaii in phaiis],
    names=detectors)
backfitters.fit(order=1)

Next we'll pass the fitters to **BackgroundRatesMatrix**, which is similar to **CountMatrix** in that it prepares background rates in the format needed by the search.

Note: passing fitters to BackgroundRatesMatrix means that background generation is agnostic to the **method**. Users are free to apply whatever fit methods they want. For example, they may want to manually tune the background applied to each detector under complicated background scenarios.

In [7]:
from background import BackgroundRatesMatrix

background = BackgroundRatesMatrix(backfitters)
bkgd_counts, bkgd_var, good = background.counts(1.728, 2.240, exposure)

print("\nBackground:")
print("  counts", bkgd_counts.reshape((len(detectors), 8)))
print("  variance", bkgd_var.reshape((len(detectors), 8)))
print("  good", good.reshape((len(detectors), 8)))


Background:
  counts [[ 35.33014426 153.63649984 101.51616545  79.90929568  75.89000211
   18.97341684  14.22981121  46.33849308]
 [ 34.79348559 173.16415835 115.59354242  81.1111437   72.11222315
   17.92896123  20.65602172  39.73354074]
 [ 31.85447768 152.20500645 107.45700619  80.80171203  76.43317845
   19.77037759  35.85838967  23.96640108]
 [ 43.39496614 200.07521497 125.05627576  86.55506462  68.51911679
   16.1946892   12.26011067  34.47496296]
 [ 35.56542423 203.91881077 128.88312036  86.14918631  68.17244615
   16.08498682  27.58935091  23.32015507]
 [ 44.94663237 203.08080925 126.31923335  83.19661277  76.50597535
   18.66615275  35.41194206  14.30135211]
 [ 34.99737801 147.94638129  94.7057319   75.98208849  68.68581186
   16.52152126  38.10135051  10.76185559]
 [ 43.75020345 184.35136587 114.56203196  81.56096597  68.47462699
   16.80863018  18.71023373  27.64631802]
 [ 34.18352823 167.63729318 104.96029428  79.5023376   68.15780768
   17.3903705   31.7210892   26.2382465

As a sanity check, let's confirm that the **CountMatrix** and **BackgroundRatesMatrix** have the same detector order and energy binning

In [8]:
print("\nSanity Checks:")
for i, det in enumerate(detectors):
    match_det = data.detectors[i] == background.detectors[i]
    match_ebounds = data.ebounds[i].low_edges() == background.ebounds[i].low_edges() \
        and data.ebounds[i].high_edges() == background.ebounds[i].high_edges()
    print("  ", det, match_det, match_ebounds)


Sanity Checks:
   n0 True True
   n1 True True
   n2 True True
   n3 True True
   n4 True True
   n5 True True
   n6 True True
   n7 True True
   n8 True True
   n9 True True
   na True True
   nb True True
   b0 True True
   b1 True True
